In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('../data/train.csv', encoding="latin8")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [47]:
df['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [24]:
df['Cabin'].value_counts()

Cabin
B96 B98        4
G6             4
C23 C25 C27    4
C22 C26        3
F33            3
              ..
E34            1
C7             1
C54            1
E36            1
C148           1
Name: count, Length: 147, dtype: int64

In [33]:
import math

class ID3:
    def __init__(self, df, target_attribute: str, attributes):
        self.df = df
        self.target_attribute = target_attribute
        self.attributes = attributes
        self.root = None
    
    def get_target_entropy(self, df: pd.DataFrame, target_atr: str)-> float:
        """
        Gets the total entropy of the dataset by summing entropy for 
        each possible value of the target concept

        param df: the entire dataframe
        param target_atr: the target concept to be examined
        returns float. The total entropy for the target_atr
        """
        att_list = df[target_atr].unique() # All possible c's that S can take
        return sum([self.calculate_entropy(df, target_atr, atr) for atr in att_list])

    
    def calculate_entropy(self, df: pd.DataFrame, target_atr: str, atr: str) -> float:
        """
        Calculates single class entropy based on the passed target_atr and atr to get the 
        entropy based on the possible value of the target_atr

        param df: the entire dataframe
        param target_atr: the column that is the target concept
        param atr: a possible value that the target_atr can take

        returns float. The entropy of the target_atr
        """
        atr_df = df[df[target_atr] == atr]
        num = len(atr_df)
        denom = len(df)

        p_pos = num / denom # pi 
        return -1 * p_pos * math.log2(p_pos) #entropy

    def information_gain(self, df: pd.DataFrame, new_attr: str, target_atr: str) -> float:
        """
        Calculates information gain based on a target attribute passed in

        param df: the entire dataframe
        param new_attr: the attribute that is going to be used to measure against the target_attr
        param target_atr: the target concept

        returns float. The information gain based on the new_attr
        """
        target_ent = self.get_target_entropy(df, target_atr) #Severity entropy: 0.89
        new_attr_values = df[new_attr].unique() # All possible values new_attr can take
        temp = 0

        for attr in new_attr_values:
            slim_df = df[df[new_attr] == attr] # Sv

            for attr_2 in slim_df[target_atr].unique():
                # Sv/S: len(slim_df)/len(df)
                temp += (len(slim_df)/len(df)) * self.calculate_entropy(slim_df, target_atr, attr_2)
        
        return target_ent - temp
    
    def split_information(self, df: pd.DataFrame, attr: str) -> float:
        """
        Split Information (Intrinsic Value) of attribute `attr` on dataset `df`.
        - Uses p_v * log2(p_v) terms.
        - Treats NaN as its own category (set dropna=True to ignore NaNs).
        """
        n = len(df)
        if n == 0:
            return 0.0  # or raise ValueError("Empty dataframe")

        # p_v for each distinct value (including NaN as a category)
        p = df[attr].value_counts(normalize=True, dropna=False)

        # SplitInfo = - sum p * log2 p  (skip p==0 just in case)
        split_info = -sum(pi * math.log2(pi) for pi in p.values if pi > 0)
        return float(split_info)


    def gain_ratio(self, df: pd.DataFrame, target_attr: str, new_attr: str) -> float:
        """
        Gain Ratio = Information Gain / Split Information.
        - Returns 0 if SplitInfo == 0 to avoid division-by-zero.
        - Optionally clamp tiny negative IGs to 0.
        """
        ig = self.information_gain(df, new_attr, target_attr)  # ensure signature matches
        # Numerical safety: IG should be >= 0 theoretically
        if ig < 0:
            ig = 0.0

        si = self.split_information(df, new_attr)
        if si == 0:
            return 0.0
        return ig / si
    

id3 = ID3(df, 'Survived', ['Ticket', 'Cabin', 'Sex'])
id3.gain_ratio(df, "Survived", "Cabin")

0.38337201160553563

In [ ]:


df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1.0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,0.0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,27.0,0,0,211536,13.0000,NaN,0.0
887,888,1,1,"Graham, Miss. Margaret Edith",1,19.0,0,0,112053,30.0000,B42,0.0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,NaN,1,2,W./C. 6607,23.4500,NaN,0.0
889,890,1,1,"Behr, Mr. Karl Howell",0,26.0,0,0,111369,30.0000,C148,1.0


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, TargetEncoder
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('../data/train.csv', encoding="latin8")

mappings = {
    "Sex": {
        "male": 0,
        "female": 1
    },
    "Embarked": {
        "S": 0,
        "C": 1,
        "Q": 2
    }
  
}

for col in df.columns:
    if col in mappings.keys():
        df[col] = df[col].map(mappings[col])

X = df.drop(columns=['Survived', 'PassengerId', "Name", "Cabin", "Ticket"])
y = df['Survived']

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
tree = DecisionTreeClassifier(
    ccp_alpha=0.1,
    criterion='gini',
    max_depth=12
)

tree.fit(X_train, y_train)
pred = tree.predict(X_test)


ValueError: could not convert string to float: 'male'